In [7]:
%load_ext autoreload
%autoreload 2

import mujoco
from swarmbots.swarm_bots_env import SwarmBotsEnv
from swarmbots.swarm.simple_swarm_tetrahedron_zx import SimpleSwarmTetrahedronZX
from swarmbots.scenarios.obstacle_street_scenario import ObstacleStreetScenario
from rendering import display_video

import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:

# swarm = SimpleSwarmCubeZX(connection_torquescale=10)
swarm = SimpleSwarmTetrahedronZX(connection_torquescale=10)
scenario = ObstacleStreetScenario(
    swarm, 
    payload_type='sphere', 
    payload_size=(0.2, 0.2, 0.2),
    payload_start_location_offset=(0, 0, 1), 
    seed=None
)

opt = mujoco.MjvOption()

env = SwarmBotsEnv(
    scenario=scenario,
    render_mode="rgb_array",
    width=640,
    height=480,
    scene_option=opt,
    action_repeat=15,
)

rng = np.random.default_rng()


frames = []
for i in range(1):
    done = False
    obs, info = env.reset()

    while not done:
        # Random action
        action = env.action_space.sample()
        action['connectors'] = True
        obs, reward, terminated, truncated, info = env.step(action)

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        done = terminated or truncated

        if info and 'error' in info:
            print('err ' + str(env.data.time))


        # if len(frames) % 50 == 0:
        #     env.data.eq_active[:] = 0
        #     env.data.eq_active[rng.integers(low=0, high=len(env.data.eq_active))] = 1

    print(f"Recorded {len(frames)} frames")
    for _ in range(15):
        frames.append(np.zeros_like(frames[0]))

# env.close()
display_video(frames, 30)

Recorded 500 frames


In [ ]:
env.observation_space

In [8]:
env.action_space

Dict('actuators': Box(-1.0, 1.0, (2, 8), float32), 'connectors': MultiBinary((2, 4)))

In [15]:
is_active, _, _ = env.swarm_connections.get_active_connections()
is_active

array([[False, False,  True, False, False, False],
       [False, False, False,  True, False, False],
       [False, False, False, False, False, False],
       [False, False, False, False, False, False],
       [False, False, False, False, False, False]])

In [26]:
action = np.asarray(env.action_space.sample()['connectors'], dtype=bool)
action

array([[ True, False,  True,  True,  True, False],
       [False, False, False, False,  True, False],
       [False,  True, False, False,  True,  True],
       [ True, False, False, False, False,  True],
       [False, False, False,  True, False,  True]])

array([[0, 1, 0, 0, 0, 1],
       [1, 1, 1, 1, 0, 1],
       [1, 0, 1, 1, 0, 0],
       [0, 1, 1, 1, 1, 0],
       [1, 1, 1, 0, 1, 0]])

In [34]:
np.stack(np.where(np.logical_and(action, np.logical_not(is_active)))).T

array([[0, 0],
       [0, 3],
       [0, 4],
       [1, 4],
       [2, 1],
       [2, 4],
       [2, 5],
       [3, 0],
       [3, 5],
       [4, 3],
       [4, 5]])

In [31]:
np.where(np.logical_and(is_active, np.logical_not(action)))

(array([1]), array([3]))

In [19]:
type(env.data.warning[mujoco.mjtWarning.mjWARN_BADQACC].lastinfo)

int

In [5]:
env.model.eq_data[:5]

array([[0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.]])

In [3]:
env.scenario.data.qpos.shape

(38,)

In [14]:
[b.name for b in env.scenario.spec.bodies]

['world',
 'Unit0--main_body',
 'Unit0--0-limb_root',
 'Unit0-ppp',
 'Unit0--0-seg2',
 'Unit0--0-connector',
 'Unit0--1-limb_root',
 'Unit0-pmm',
 'Unit0--1-seg2',
 'Unit0--1-connector',
 'Unit0--2-limb_root',
 'Unit0-mpm',
 'Unit0--2-seg2',
 'Unit0--2-connector',
 'Unit0--3-limb_root',
 'Unit0-mmp',
 'Unit0--3-seg2',
 'Unit0--3-connector',
 'Unit1--main_body',
 'Unit1--0-limb_root',
 'Unit1-ppp',
 'Unit1--0-seg2',
 'Unit1--0-connector',
 'Unit1--1-limb_root',
 'Unit1-pmm',
 'Unit1--1-seg2',
 'Unit1--1-connector',
 'Unit1--2-limb_root',
 'Unit1-mpm',
 'Unit1--2-seg2',
 'Unit1--2-connector',
 'Unit1--3-limb_root',
 'Unit1-mmp',
 'Unit1--3-seg2',
 'Unit1--3-connector',
 'Wall_0_Left',
 'Wall_0_Right',
 'Ramp_0',
 'Wall_1_Left',
 'Wall_1_Right',
 'Ramp_1',
 'Wall_2_Left',
 'Wall_2_Right',
 'Ramp_2',
 'Wall_3_Left',
 'Wall_3_Right',
 'Ramp_3',
 'Wall_4_Left',
 'Wall_4_Right',
 'Ramp_4']

In [15]:
env.scenario.spec.body('Unit0--main_body')

[autoreload of swarmbots.scenarios.base_scenario failed: Traceback (most recent call last):
  File "C:\Users\Brn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\extensions\autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "C:\Users\Brn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\extensions\autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 936, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1074, in get_code
  File "<f